In [49]:
import csv
import numpy as np
import sys
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

In [13]:
import urllib.request
urllib.request.urlretrieve(
    'https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data',
    'processed.cleveland.data'
)

('processed.cleveland.data', <http.client.HTTPMessage at 0x16a0f0a10>)

In [15]:
datafilename = 'processed.cleveland.data' # input filename
age      = 0                              # column indexes in input file
sex      = 1
cp       = 2
trestbps = 3
chol     = 4
fbs      = 5
restecg  = 6
thalach  = 7
exang    = 8
oldpeak  = 9
slope    = 10
ca       = 11
thal     = 12
num      = 14 # this is the thing we are trying to predict

# Since feature names are not in the data file, we code them here
feature_names = [ 'age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'num' ]

num_samples = 303 # size of the data file. 
num_features = 13

#
# Open and read data file in csv format
#
# After processing:
# 
# data   is the variable holding the features;
# target is the variable holding the class labels.

try:
    with open( datafilename ) as infile:
        indata = csv.reader( infile )
        data = np.empty(( num_samples, num_features ))
        target = np.empty(( num_samples,), dtype=int )
        i = 0
        for j, d in enumerate( indata ):
            ok = True
            for k in range(0,num_features): # If a feature has a missing value
                if ( d[k] == "?" ):         # we don't use that record.
                    ok = False
            if ( ok ):
                data[i] = np.asarray( d[:-1], dtype=np.float64 )
                target[i] = np.asarray( d[-1], dtype=int )
                i = i + 1
except IOError as iox:
    print('there was an I/O error trying to open the data file: ' + str( iox ))
    sys.exit()
except Exception as x:
    print('there was an error: ' + str( x ))
    sys.exit()

# Here is are the sets of features:
data
# Here is the diagnosis for each set of features:
target

# How many records do we have?
num_samples = i
print("Number of samples:", num_samples)

# Adjust the size of data and target so that they only hold the values
# loaded from the CSV file
#
# This, elegant approach, is due to Adrian Salazar Gomez (2018/19):

data = data[:num_samples]
target = target[:num_samples]

Number of samples: 297


In [29]:
df = pd.DataFrame(data, columns=feature_names[:-1])

In [31]:
df

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
292,57.0,0.0,4.0,140.0,241.0,0.0,0.0,123.0,1.0,0.2,2.0,0.0,7.0
293,45.0,1.0,1.0,110.0,264.0,0.0,0.0,132.0,0.0,1.2,2.0,0.0,7.0
294,68.0,1.0,4.0,144.0,193.0,1.0,0.0,141.0,0.0,3.4,2.0,2.0,7.0
295,57.0,1.0,4.0,130.0,131.0,0.0,0.0,115.0,1.0,1.2,2.0,1.0,7.0


In [33]:
target

array([0, 2, 1, 0, 0, 0, 3, 0, 2, 1, 0, 0, 2, 0, 0, 0, 1, 0, 0, 0, 0, 0,
       1, 3, 4, 0, 0, 0, 0, 3, 0, 2, 1, 0, 0, 0, 3, 1, 3, 0, 4, 0, 0, 0,
       1, 4, 0, 4, 0, 0, 0, 0, 2, 0, 1, 1, 1, 1, 0, 0, 2, 0, 1, 0, 2, 2,
       1, 0, 2, 1, 0, 3, 1, 1, 1, 0, 1, 0, 0, 3, 0, 0, 0, 3, 0, 0, 0, 0,
       0, 0, 3, 0, 0, 0, 1, 2, 3, 0, 0, 0, 0, 0, 0, 3, 0, 2, 1, 2, 3, 1,
       1, 0, 2, 2, 0, 0, 0, 3, 2, 3, 4, 0, 3, 1, 0, 3, 3, 0, 0, 0, 0, 0,
       0, 0, 0, 4, 3, 1, 0, 0, 1, 0, 1, 0, 1, 4, 0, 0, 0, 0, 0, 0, 4, 3,
       1, 1, 1, 2, 0, 0, 4, 0, 0, 0, 0, 0, 1, 0, 3, 0, 1, 0, 4, 1, 0, 1,
       0, 0, 3, 2, 0, 0, 1, 0, 0, 2, 1, 2, 0, 3, 2, 0, 3, 0, 0, 0, 1, 0,
       0, 0, 0, 0, 3, 3, 3, 0, 1, 0, 4, 0, 3, 1, 0, 0, 0, 0, 0, 0, 0, 0,
       3, 1, 0, 0, 0, 3, 2, 0, 2, 1, 0, 0, 3, 2, 1, 0, 0, 0, 0, 0, 2, 0,
       2, 2, 1, 3, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 3, 0, 0, 4, 2, 2,
       1, 0, 1, 0, 2, 0, 1, 0, 0, 0, 1, 0, 2, 0, 3, 0, 2, 4, 2, 0, 0, 1,
       0, 2, 2, 1, 0, 3, 1, 1, 2, 3, 1])

In [37]:
len(target)

297

In [83]:
from sklearn.model_selection import train_test_split
from sklearn import tree

# 1. Split the data 80/20
data_train, data_test, target_train, target_test = train_test_split(
    data, target, test_size=0.2, random_state=42
)

# 2. Build and test a decision tree
DT = tree.DecisionTreeClassifier()
DT.fit(data_train, target_train)

print("Training accuracy:", DT.score(data_train, target_train))
print("Test accuracy:", DT.score(data_test, target_test))

Training accuracy: 1.0
Test accuracy: 0.5


In [89]:
# The discrete features we're using
discrete_features = [sex, cp, fbs, restecg, exang, slope, ca]

classes = np.unique(target_train)

# Step 1: Prior probabilities p(num = c)
priors = {}
for c in classes:
    count = sum(1 for t in target_train if t == c)
    priors[c] = count / len(target_train)

# Step 2: Conditional probabilities p(feature = v | num = c)
conditionals = {}
for f in discrete_features:
    conditionals[f] = {}
    feature_values = np.unique(data_train[:, f])
    for c in classes:
        conditionals[f][c] = {}
        class_count = sum(1 for t in target_train if t == c)
        for v in feature_values:
            count = sum(1 for i in range(len(target_train))
                        if target_train[i] == c and data_train[i][f] == v)
            conditionals[f][c][v] = count / class_count

# Step 3: Predict for a single example
def predict(sample):
    best_class = None
    best_prob = -1
    for c in classes:
        prob = priors[c]
        for f in discrete_features:
            v = sample[f]
            if v in conditionals[f][c]:
                prob *= conditionals[f][c][v]
            else:
                prob *= 0
        if prob > best_prob:
            best_prob = prob
            best_class = c
    return best_class

# Step 4: Score on test data
correct = sum(1 for i in range(len(target_test)) 
              if predict(data_test[i]) == target_test[i])
accuracy = correct / len(target_test)
print("Test accuracy:", accuracy)

Test accuracy: 0.65


In [91]:
from sklearn.naive_bayes import MultinomialNB

# Select only the discrete features
data_train_discrete = data_train[:, discrete_features]
data_test_discrete = data_test[:, discrete_features]

# Train and test
MNB = MultinomialNB()
MNB.fit(data_train_discrete, target_train)

print("Test accuracy:", MNB.score(data_test_discrete, target_test))

Test accuracy: 0.6


In [115]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score
from sklearn.preprocessing import StandardScaler

# Load the data - StoneFlakes.dat is space separated
df = pd.read_csv('/Users/benvarvill/Downloads/StoneFlakes.dat',header=0)

In [117]:
print(df.head())
print(df.shape)
print(df.describe())

                                        ID    LBI   RTI  WDI FLA  PSF  FSF ZDF1 PROZD
ar      ? 35.3 2.60   ? 42.4 24.2 47.1                                             69
arn  1.23 27.0 3.59 122  0.0 40.0 40.0                                             30
be   1.24 26.5 2.90 121 16.0 20.7 29.7                                             72
bi1  1.07 29.1 3.10 114 44.0  2.6 26.3                                             68
bi2  1.08 43.7 2.40 105 32.6  5.8 10.7                                             42
(79, 1)
       ID    LBI   RTI  WDI FLA  PSF  FSF ZDF1 PROZD
count                                      79.000000
mean                                       75.088608
std                                        15.255255
min                                        30.000000
25%                                        65.500000
50%                                        78.000000
75%                                        87.000000
max                                        98.000000